# Demonstration notebook for generation with SurvivEHR

In this notebook we demonstrate how a pre-trained model can be used for generation of future patient timelines

In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

!pwd

%load_ext autoreload
%autoreload 2
%env SLURM_NTASKS_PER_NODE=28       # TODO: define an env variable to fix for a local hpc environment issue, this shouldn't be needed

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/modelling/SurvivEHR/notebooks/CompetingRisk/0_pretraining/2_generation
env: SLURM_NTASKS_PER_NODE=28       # TODO: define an env variable to fix for a local hpc environment issue, this shouldn't be needed


In [8]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import logging
from pycox.evaluation import EvalSurv
from tqdm import tqdm
from hydra import compose, initialize
from omegaconf import OmegaConf
from contextlib import redirect_stdout
import pandas as pd
from tqdm import tqdm
import warnings

from SurvivEHR.examples.modelling.SurvivEHR.run_experiment import run
from SurvivEHR.examples.modelling.SurvivEHR.setup_causal_experiment import CausalExperiment
from SurvivEHR.src.models.survival.task_heads.causal import SurvStreamGPTForCausalModelling

from FastEHR.dataloader import FoundationalDataModule
from FastEHR.database.collector import SQLiteDataCollector

torch.manual_seed(1337)
torch.set_float32_matmul_precision('medium')
warnings.simplefilter('error', np.VisibleDeprecationWarning)

logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.ERROR)          # Remove deprecation warnings from older dataset format

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"    # if more informative debugging statements are needed
print(f"Using device: {device}.")

Using device: cuda.


# Generation from real prompts

### Create utility functions

In [4]:
def batch_to_sample(batch, idx, remove_masked=True):
    # Take `idx` patient from this batch
    batch = {k: v[[idx]] for k, v in batch.items()}

    if remove_masked:
        mask = (batch["tokens"][0, :] != 0)
        batch["tokens"] = batch["tokens"][:, mask]
        batch["ages"] = batch["ages"][:, mask]
        batch["values"] = batch["values"][:, mask]

    return batch

def clip_outliers(token, unstandardised_value):
    """
    Because of heavy right tails in the value distributions, standardisation of some token values over-estimates the lower quantile.
    This method ensures no values beyond those seen in the data are reported.
    """
    try:
        assert not np.isnan(unstandardised_value)
        token_meta = dm.meta_information["measurement_tables"][dm.meta_information["measurement_tables"]["event"] == token]
        _min = token_meta["min"]
        _max = token_meta["max"]
        unstandardised_value = np.min((unstandardised_value, _max))
        unstandardised_value = np.max((unstandardised_value, _min))
    except:
        pass
        
    return unstandardised_value

def report_generation(static, tokens, ages, values, attention_mask, true_seq_len, dm, eos_token="DEATH", **kwargs):
    """
    """
    tokens = tokens[0, :]
    ages = ages[0, :]
    values = values[0, :]
    attention_mask = attention_mask[0, :]

    static = dm.test_set._decode_covariates(static.cpu())
    print("STATIC INFORMATION")
    print("="*120)
    for key, item in static.items():
        print(f"\t{key}:".ljust(20) + f"{item[0]}")

    # Report
    tokens = dm.tokenizer.decode(tokens.tolist()).split(" ")
    diagnoses = []
    last_age_day, last_age_week = 0, 0
    print("\n\nGiven patient context".upper())
    print("="*120)
    print(f"\tEVENT".ljust(75) + "| AGE IN WEEKS (days, years)".ljust(30) + " | VALUE")
    for idx_event, (token, _age, value, attn_mask) in enumerate(zip(tokens, ages, values, attention_mask)):

        if attn_mask == 0:
            break
            
        # Unscale age and bin to week fidelity
        age_day = int(_age * dm.test_set.time_scale)
        age_week = int(age_day / 7) 
        age_years = int(age_day / 365)

        # If new event create break
        if age_week != last_age_week:
            print("\t" + "."*60 + "new week" + "."*60)

        # Report next event
        age = f"{age_week}\t ({age_day}, {age_years})"
        unstandardised_value = clip_outliers(token, dm.unstandardise(token, value))
        value = f"{unstandardised_value:.2f}".ljust(10) + f"({value:.2f})"
        print(f"\t{token.ljust(75)}| {age.ljust(30)}| {value}".ljust(20))
        
        if token.upper() == token:
            diagnoses.append(token)

        if idx_event == true_seq_len - 1:
            print("\n" + "="*120)
            print("Diagnosis summary".upper())
            print(f"{diagnoses}")
            print("="*120)
            print("\n")
            print("Predicted future events".upper())
            print("="*120)
            print(f"\tEVENT".ljust(75) + "| AGE IN WEEKS (days, years)".ljust(30) + "| VALUE")


        last_age_day = age_day
        last_age_week = age_week
        if token == eos_token:
            break

def log_generation(tokens, ages, values, attention_mask, observed_seq_lens, dm, eos_token="DEATH"):

    data = []
    
    for patient_idx in range(tokens.shape[0]):

        # patient_observed_seq_len
        observed_seq_len = observed_seq_lens[patient_idx] - 1
        
        # Remove the prompt context at the start of the generation
        generated_tokens = tokens[patient_idx, observed_seq_len:].cpu().numpy()
        generated_ages   = ages[patient_idx, observed_seq_len:].cpu().numpy()
        generated_values = values[patient_idx, observed_seq_len:].cpu().numpy()
        generated_attention_mask = attention_mask[patient_idx, observed_seq_len:].cpu().numpy()
    
        for generation_step in range(generated_tokens.shape[0]-1):

            # If we reached padding then stop
            if generated_attention_mask[generation_step+1] == 0:
                break
            
            # Event transition
            previous_token = dm.decode([generated_tokens[generation_step]])
            next_token = dm.decode([generated_tokens[generation_step+1]])
            
            # Age, unscaled to years old
            next_age = generated_ages[generation_step+1]
            age_day = int(next_age * dm.test_set.time_scale)
            age_week = int(age_day / 7) 
            age_years = int(age_day / 365)

            # Ignore any events occuring after terminating token
            if previous_token == eos_token:
                break

            # Log transition, and how many steps into generation this occurred. 
            record = [previous_token, next_token, generation_step, age_years]
            data.append(record)

    return data

In [5]:
pre_trained_models = ["SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1"]
config_names = ["config_CompetingRisk11M"]

# Generate future patient timelines

For a handful of selected patients from each dataset, generate potential future trajectories and save the outputs to file for analysis.

This is similar to ``1_print_generation.ipynb``, but saves outputs to file and considers multiple datasets.

In [11]:
datasets = [ "PreTrain", "FineTune_Hypertension", "FineTune_CVD", "FineTune_MultiMorbidity50+"]
patients_of_interest = [None,[0],[10],[1]]

for pre_trained_model, config_name in zip(pre_trained_models, config_names):
    os.makedirs(f"figs/generation/{pre_trained_model}/", exist_ok=True) 

    # load the configuration file, override any settings 
    with initialize(version_base=None, config_path="../../../../confs", job_name="testing_notebook"):
        cfg = compose(config_name=config_name, 
                      overrides=[# Experiment setup
                                 f"experiment.run_id='{pre_trained_model}'",
                                 "experiment.train=False",
                                 "experiment.test=False",
                                 "experiment.log=False",
                                 # Dataloader
                                 "data.meta_information_path=/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle",
                                 "data.min_workers=3",
                                ]
                     )     
    experiment, dm = run(cfg)     
    print(f"Loaded model {pre_trained_model} with {sum(p.numel() for p in experiment.parameters())/1e6} M parameters")
    
    for idx_dataset, dataset in enumerate(datasets):
        print(f"Generating patient timelines for dataset {dataset}")
        
        gen_save_path = f'figs/generation/{pre_trained_model}/{dataset}_dataset/'
        os.makedirs(gen_save_path, exist_ok=True) 
    
        # Load dataset
        dm = FoundationalDataModule(path_to_db=cfg.data.path_to_db,
                                    path_to_ds=f"/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/{dataset}/",
                                    overwrite_meta_information=cfg.data.meta_information_path,
                                    load=True,
                                    supervised=False if dataset.lower()=="pretrain" else True,
                                    )
        
        # Load the first batch
        for batch in dm.test_dataloader():

            # Put all items on gpu
            batch = {k: v.to(device) for k, v in batch.items()}

            # # Optionally, get only the samples of interest from batch 
            # #       (shuffle is by default turned off in test data, so this should align to dm.test_set[idx])
            # batch = batch_to_sample(batch, 6, remove_masked=True)
            break
        
        for idx_gen in range(5):

            # Generate forward
            tokens, ages, values, attention_mask = experiment.model.generate(
                static_covariates=batch["static_covariates"],
                tokens=batch['tokens'],
                ages=batch['ages'],
                values=batch['values'],
                attention_mask=batch['attention_mask'],
                max_new_tokens=50,
                exceed_block_size=True,
                )

            # Report generated outcomes
            patients = patients_of_interest[idx_dataset] if patients_of_interest[idx_dataset] is not None else [i for i in range(tokens.shape[0])]
            for idx_patient in tqdm(patients, ascii=True, desc=f"Reporting generation {idx_gen + 1} results for all patients in batch"):
                
                out_dir = gen_save_path + f'patient{idx_patient}/'
                os.makedirs(out_dir, exist_ok=True)
                with open(out_dir + f"generation{idx_gen}.txt", 'w') as f:
                    with redirect_stdout(f):
                        report_generation(
                            static         = batch["static_covariates"][idx_patient], 
                            tokens         = tokens[[idx_patient],:],
                            ages           = ages[[idx_patient],:], 
                            values         = values[[idx_patient], :], 
                            attention_mask = attention_mask[[idx_patient], :],
                            true_seq_len   = batch["attention_mask"][[idx_patient],:].sum(), 
                            dm             = dm
                        )
                        
            # Log for plotting
            # log_generation(tokens, ages, values, observed_seq_len, dm)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Loaded model SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1 with 11.20919 M parameters
Generating patient timelines for dataset PreTrain


Reporting generation 1 results for all patients in batch: 100%|##########| 64/64 [00:06<00:00,  9.89it/s]
Reporting generation 2 results for all patients in batch: 100%|##########| 64/64 [00:06<00:00, 10.25it/s]
Reporting generation 3 results for all patients in batch: 100%|##########| 64/64 [00:06<00:00, 10.15it/s]
Reporting generation 4 results for all patients in batch: 100%|##########| 64/64 [00:06<00:00, 10.26it/s]
Reporting generation 5 results for all patients in batch: 100%|##########| 64/64 [00:06<00:00, 10.24it/s]


Generating patient timelines for dataset FineTune_Hypertension


Reporting generation 1 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 22.98it/s]
Reporting generation 2 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 24.28it/s]
Reporting generation 3 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 25.91it/s]
Reporting generation 4 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 22.95it/s]
Reporting generation 5 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 29.01it/s]


Generating patient timelines for dataset FineTune_CVD


Reporting generation 1 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 14.49it/s]
Reporting generation 2 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 16.87it/s]
Reporting generation 3 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 23.40it/s]
Reporting generation 4 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 22.01it/s]
Reporting generation 5 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00, 21.66it/s]


Generating patient timelines for dataset FineTune_MultiMorbidity50+


Reporting generation 1 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00,  5.69it/s]
Reporting generation 2 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00,  5.55it/s]
Reporting generation 3 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00,  6.66it/s]
Reporting generation 4 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00,  5.79it/s]
Reporting generation 5 results for all patients in batch: 100%|##########| 1/1 [00:00<00:00,  5.95it/s]


# Transition probability from generations

Generate patient timelines for many test patients and condense into files for plotting

In [12]:
generations_per_patient = 5
datasets = [ "FineTune_MultiMorbidity50+", "FineTune_CVD", "FineTune_Hypertension", "PreTrain",]
# fraction_of_batches = 1.0  # 0.035
number_of_batches = 1000

for pre_trained_model, config_name in zip(pre_trained_models, config_names):
    os.makedirs(f"figs/generation/{pre_trained_model}/", exist_ok=True) 

    # load the configuration file, override any settings 
    with initialize(version_base=None, config_path="../../../../confs", job_name="testing_notebook"):
        cfg = compose(config_name=config_name, 
                      overrides=[# Experiment setup
                                 f"experiment.run_id='{pre_trained_model}'",
                                 "experiment.train=False",
                                 "experiment.test=False",
                                 "experiment.log=False",
                                 # Dataloader
                                 "data.meta_information_path=/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle",
                                 "data.min_workers=3",
                                ]
                     )     
    
    experiment, dm = run(cfg)     
    print(f"Loaded model {pre_trained_model} with {sum(p.numel() for p in experiment.parameters())/1e6} M parameters")
    
    for idx_dataset, dataset in enumerate(datasets):
        print(f"Generating patient timelines for dataset {dataset}")
        
        # Create save path
        gen_save_path = f'figs/generation/{pre_trained_model}/{dataset}_dataset/'
        os.makedirs(gen_save_path, exist_ok=True) 

        # Load dataset
        dm = FoundationalDataModule(path_to_db=cfg.data.path_to_db,
                                    path_to_ds=f"/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/{dataset}/",
                                    overwrite_meta_information=cfg.data.meta_information_path,
                                    load=True,
                                    supervised=False if dataset.lower()=="pretrain" else True,
                                    )
        
        dataset_generated_data = []
        # number_of_batches = int(fraction_of_batches * len(dm.test_dataloader())) 
        number_of_batches = np.min((number_of_batches, len(dm.test_dataloader())))
        for idx_batch, batch in tqdm(enumerate(dm.test_dataloader()), total=number_of_batches):
            
            if idx_batch > number_of_batches:
                break

            # Put all items on gpu
            batch = {k: v.to(device) for k, v in batch.items()}

            for idx_gen in range(generations_per_patient):

                # Generate forward
                tokens, ages, values, attention_mask = experiment.model.generate(
                    static_covariates=batch["static_covariates"],
                    tokens=batch['tokens'],
                    ages=batch['ages'],
                    values=batch['values'],
                    attention_mask=batch['attention_mask'],
                    max_new_tokens=5,
                    exceed_block_size=True,
                    )                

                # Report generated outcomes for one patient
                # idx_patient = 0
                # report_generation(
                #     static         = batch["static_covariates"][idx_patient], 
                #     tokens         = tokens[[idx_patient],:],
                #     ages           = ages[[idx_patient],:], 
                #     values         = values[[idx_patient], :], 
                #     attention_mask = attention_mask[[idx_patient], :],
                #     true_seq_len   = batch["attention_mask"][[idx_patient],:].sum(), 
                #     dm             = dm
                #     )
                
                # Log for plotting                
                observed_seq_lens = batch["attention_mask"].sum(axis=1).long().tolist() 
                gen_data = log_generation(tokens, ages, values, attention_mask, observed_seq_lens, dm)
                
                # Record
                dataset_generated_data.append(gen_data)

        dataset_generated_data = np.concatenate(dataset_generated_data)
        dataset_generated_data = pd.DataFrame(dataset_generated_data, columns=["Previous event", "Next event", "Generation step", "Age (years)"])
        dataset_generated_data.to_csv(gen_save_path + f'next_event_{dataset}.csv', index=False)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Loaded model SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1 with 11.20919 M parameters
Generating patient timelines for dataset FineTune_MultiMorbidity50+


 11%|█         | 111/1000 [11:15<1:30:06,  6.08s/it]

KeyboardInterrupt



In [15]:
# # Partial completion print
# dataset_generated_data = np.concatenate(dataset_generated_data)
# dataset_generated_data = pd.DataFrame(dataset_generated_data, columns=["Previous event", "Next token", "Generation step", "Age (years)"])
# display(dataset_generated_data.head())
# print(len(dataset_generated_data))

,Previous event,Next token,Generation step,Age (years)
0,ACE_Inhibitors_D2T,HYPERTENSION,0,48
1,HYPERTENSION,Basophil_count_22,1,48
2,Basophil_count_22,Diastolic_blood_pressure_5,2,48
3,Diastolic_blood_pressure_5,GFR_calculated_abbreviated_MDRD_34,3,49
4,GFR_calculated_abbreviated_MDRD_34,Basophil_count_22,4,49


177277


# Appendix: Misc reference data for paper

## How prevalent is anxiety relative other event types?

In [19]:
import polars as pl 
print(dm.tokenizer._event_counts["FREQUENCY"].sum())
print(dm.tokenizer._event_counts)

anxiety_freq = dm.tokenizer._event_counts.filter(pl.col("EVENT") == "ANXIETY")["FREQUENCY"].item()
display(anxiety_freq)

# display([i for i in dm.tokenizer._event_counts["EVENT"] if i.upper() == i ])
display(len(dm.tokenizer._event_counts.filter(pl.col("FREQUENCY") >= anxiety_freq)) - 1)  # remove UNK token
display(len(dm.tokenizer._event_counts.filter(pl.col("FREQUENCY") < anxiety_freq)))

1.0
shape: (264, 3)
┌──────────────────────────────┬───────────┬───────────┐
│ EVENT                        ┆ COUNT     ┆ FREQUENCY │
│ ---                          ┆ ---       ┆ ---       │
│ str                          ┆ u32       ┆ f64       │
╞══════════════════════════════╪═══════════╪═══════════╡
│ UNK                          ┆ 0         ┆ 0.0       │
│ ADDISONS_DISEASE             ┆ 6691      ┆ 8.8559e-7 │
│ CYSTICFIBROSIS               ┆ 7053      ┆ 9.3350e-7 │
│ SYSTEMIC_SCLEROSIS           ┆ 8772      ┆ 0.000001  │
│ …                            ┆ …         ┆ …         │
│ Diastolic_blood_pressure_5   ┆ 241026671 ┆ 0.031901  │
│ Systolic_blood_pressure_4    ┆ 241541988 ┆ 0.031969  │
│ Statins                      ┆ 249911634 ┆ 0.033077  │
│ Lipid_lowering_drugs_Optimal ┆ 263876709 ┆ 0.034926  │
└──────────────────────────────┴───────────┴───────────┘


0.0004713146624491795

138

125

## Code to check samples from dataset against the SQLite database

In [20]:
# path_to_directory = os.getcwd() + "/../data/"
PATH_TO_DB = "/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/cprd.db"

collector = SQLiteDataCollector(db_path=PATH_TO_DB)
collector.connect()

In [21]:
collector.cursor.execute("""SELECT name FROM sqlite_master WHERE type='table' LIMIT 3;""")   # 
results = collector.cursor.fetchall()
for result in results:
    print(result)

('static_table',)
('diagnosis_table',)
('measurement_25_Hydroxyvitamin_D2_level_92',)


### Search for a patient based on known dataset baseline covariates

This is the smallest table, and indexed such that searches are very fast, so we can start the search here to narrow the pool.

In [22]:
collector.cursor.execute("""SELECT * FROM static_table WHERE sex=='F' AND imd=='4' AND ethnicity=='WHITE' AND YEAR_OF_BIRTH LIKE '1933-%'""")   # 

patient_ids = []
results = collector.cursor.fetchall()
for result in results:
    # print(result)
    patient_ids.append(result[1])
patient_ids_str1 = ", ".join(str(pid) for pid in patient_ids)
print(f"{len(patient_ids)} patients with static match")

6884 patients with static match


### Of these, search for patient based on known dataset historical diagnoses

In [23]:
# query = f"""
#             SELECT DISTINCT PATIENT_ID
#             FROM diagnosis_table
#             WHERE EVENT IN ('HYPERTENSION', 'ANY_DEAFNESS_HEARING_LOSS_V2', 'IHDINCLUDINGMI_OPTIMALV2, OSTEOARTHRITIS,TYPE2DIABETES') 
#                 AND patient_id IN ({patient_ids_str})
#             GROUP BY patient_id
#             HAVING COUNT(DISTINCT event) >= 5
#             """

query = f"""SELECT patient_id
            FROM diagnosis_table
            WHERE patient_id IN ({patient_ids_str1} )
            GROUP BY patient_id
            HAVING 
                 COUNT(
                      DISTINCT CASE WHEN event IN ('HYPERTENSION', 'ANY_DEAFNESS_HEARING_LOSS_V2', 'IHDINCLUDINGMI_OPTIMALV2', 'OSTEOARTHRITIS', 'TYPE2DIABETES')
                                    THEN event
                               END
                    ) = 5
            ORDER BY patient_id;
            """

patient_ids = []
collector.cursor.execute(query)   # measurement_ACE_Inhibitors_D2T
results = collector.cursor.fetchall()
for result in results:
    patient_ids.append(result[0])
patient_ids_str2 = ", ".join(str(pid) for pid in patient_ids)
print(f"{len(patient_ids)} patients with static match and all of these events")

114 patients with static match and all of these events


### View remaining patients

For each of these patients, print the diagnosis events in the order they occurred (changing date to year).

In [33]:
patient_ids = []
collector.cursor.execute(f"""
    SELECT patient_id, practice_id, event, strftime('%Y', date) as year
    FROM diagnosis_table 
    WHERE patient_id IN ({patient_ids_str2}) 
    ORDER BY patient_id ASC, date ASC
""")

results = collector.cursor.fetchall()
for result in results:
    print(result)
    patient_ids.append(result[1])

# collector.cursor.execute("""SELECT * FROM diagnosis_table WHERE event=='HYPERTENSION' AND date LIKE '2003-%' LIMIT 10""")   # measurement_ACE_Inhibitors_D2T
# results = collector.cursor.fetchall()
# for result in results:
#     print(result)


(67978220368, 20368, 'ASTHMA_PUSHASTHMA', '1998')
(67978220368, 20368, 'PMRANDGCA', '2000')
(67978220368, 20368, 'OSTEOARTHRITIS', '2000')
(67978220368, 20368, 'ANY_DEAFNESS_HEARING_LOSS_V2', '2000')
(67978220368, 20368, 'COPD', '2001')
(67978220368, 20368, 'TYPE2DIABETES', '2002')
(67978220368, 20368, 'HYPERTENSION', '2002')
(67978220368, 20368, 'IHDINCLUDINGMI_OPTIMALV2', '2004')
(67978220368, 20368, 'AF', '2007')
(67978220368, 20368, 'CKDSTAGE3TO5', '2008')
(67978220368, 20368, 'VALVULARDISEASES_V2', '2009')
(67978220368, 20368, 'DEATH', '2010')
(73302920502, 20502, 'DEPRESSION', '2007')
(73302920502, 20502, 'PREVALENT_IBS_V2', '2008')
(73302920502, 20502, 'HYPERTENSION', '2008')
(73302920502, 20502, 'TYPE2DIABETES', '2008')
(73302920502, 20502, 'ANXIETY', '2008')
(73302920502, 20502, 'OSTEOARTHRITIS', '2009')
(73302920502, 20502, 'IHDINCLUDINGMI_OPTIMALV2', '2012')
(73302920502, 20502, 'MINFARCTION', '2015')
(73302920502, 20502, 'ANY_DEAFNESS_HEARING_LOSS_V2', '2015')
(73302920502,

### Get specific records of an identified patient

In [34]:
patient_ids_str = ", ".join(str(pid) for pid in patient_ids)
collector.cursor.execute(f"""SELECT * FROM measurement_Systolic_blood_pressure_4 WHERE patient_id == 2666145020970""")   # 5437879821203
results = collector.cursor.fetchall()
for result in results:
    print(result)

(20970, 2666145020970, 'Systolic_blood_pressure_4', 142.0, '2012-08-14')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 130.0, '2016-07-12')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 139.0, '2017-07-04')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 169.0, '2018-02-26')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 163.0, '2018-03-05')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 174.0, '2018-03-05')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 154.0, '2018-11-15')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 168.0, '2018-11-22')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 134.0, '2019-08-16')
(20970, 2666145020970, 'Systolic_blood_pressure_4', 142.0, '2021-12-02')


In [35]:
collector.cursor.execute("""SELECT * FROM static_table WHERE practice_id=='21573' AND patient_id=='6626432621573'""")   # 

results = collector.cursor.fetchall()
for result in results:
    print(result)


collector.cursor.execute("""SELECT * FROM measurement_Body_mass_index_3 WHERE practice_id=='21573' AND patient_id=='6626432621573'""")   # 

results = collector.cursor.fetchall()
for result in results:
    print(result)

(21573, 6626432621573, 'ASIAN', '2000-07-15', 'F', 'E', 4, 'North West', '2019-12-13', '2019-12-13', '2021-09-21')
(21573, 6626432621573, 'Body_mass_index_3', 17.6, '2018-12-13')
